In [ ]:
# RQ5: Sensitivity to Evaluation Metrics
# How does model ranking change when different evaluation metrics are considered?

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/marketing-and-product-performance-dataset/marketing_and_product_performance.csv')
for col in ['Subscription_Tier', 'Common_Keywords']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df = df.drop(columns=['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID'])
X = df.drop(columns=['Units_Sold'])
y = df['Units_Sold']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
models = {
    'Lin. Reg.': LinearRegression(),
    'Dec. Tree': DecisionTreeRegressor(random_state=42),
    'K-NN': KNeighborsRegressor(n_neighbors=5),
    'Rand. Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='rbf')
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    results.append({'Model': name,
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2': r2_score(y_test, preds)})

res_df = pd.DataFrame(results)
res_df['Rank_MAE'] = res_df['MAE'].rank()
res_df['Rank_RMSE'] = res_df['RMSE'].rank()
res_df['Rank_R2'] = res_df['R2'].rank(ascending=False)
print(res_df[['Model','MAE','RMSE','R2','Rank_MAE','Rank_RMSE','Rank_R2']])
res_df.to_csv('RQ5_metrics_sensitivity.csv', index=False)

In [ ]:
rank_data = res_df[['Rank_MAE','Rank_RMSE','Rank_R2']].values
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(rank_data, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks([0,1,2])
ax.set_xticklabels(['MAE Rank','RMSE Rank','R² Rank'], fontsize=12)
ax.set_yticks(range(len(res_df)))
ax.set_yticklabels(res_df['Model'], fontsize=11)
for i in range(len(res_df)):
    for j in range(3):
        ax.text(j, i, f'{int(rank_data[i,j])}', ha='center', va='center', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Rank (1=best)')
ax.set_title('RQ5: Ranking Sensitivity Across Evaluation Metrics', fontweight='bold')
plt.tight_layout()
plt.savefig('RQ5_metrics_sensitivity.pdf', dpi=150, bbox_inches='tight')
plt.show()